# 02 — Dark Web Listing Classification

**Goal:** classify dark web marketplace listings into the non-drug categories that matter for CTI — **Services, Data, Info, Counterfeits, Forgeries, Weapons, Electronics**, etc. — using ModernBERT on GPU.

## Why this notebook exists

CTI 301 nb 01 taught ModernBERT to read *full threat reports* (2k–8k tokens). That's one end of the spectrum: long, carefully written, analyst-facing text. This notebook is the **other end**: short, noisy, adversarially written listings from an actual dark web marketplace. Same model family, very different text.

- **Dataset:** Agora marketplace dump (2014–2015, ~110k listings, CC0 on Kaggle). Classic research dataset — dated but openly redistributable, which most dark-web corpora are not.
- **Filter:** drop the 85% of listings that are drugs. What's left — fraud services, stolen data, credentials, counterfeits, weapons — is what SOC analysts actually care about.
- **Model:** `answerdotai/ModernBERT-base` on GPU. Listings are short (median 38 words) so we use `MAX_LEN=256`, batch 16, fp16 — fast.

## Prerequisites

```bash
pip install -U transformers datasets scikit-learn torch pandas kaggle
```

Kaggle API credentials at `~/.kaggle/kaggle.json` (chmod 600). Free Kaggle account required — see https://www.kaggle.com/settings/account → API → Create New Token.

## 1 — Download + build the dataset

Pull the Agora CSV from Kaggle, drop drugs, merge `Info`/`Information` duplicates, build a cleaned multi-class dataset.

In [1]:
import json, subprocess
from pathlib import Path
import numpy as np, pandas as pd
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel

DATA = Path('data'); DATA.mkdir(exist_ok=True)
PROCESSED = Path('processed'); PROCESSED.mkdir(exist_ok=True)
CSV = DATA / 'Agora.csv'

if not CSV.exists():
    print('Downloading Agora dataset from Kaggle...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'philipjames11/dark-net-marketplace-drug-data-agora-20142015',
                    '--unzip', '-p', str(DATA)], check=True)

# Latin-1 — this dump has mixed encodings from forum scrape
df = pd.read_csv(CSV, encoding='latin-1', on_bad_lines='skip')
df.columns = [c.strip() for c in df.columns]  # column names have leading spaces
print(f'Raw rows: {len(df)}')

# --- Filter to CTI-relevant top-level categories -----------------------
# Drop Drugs (85% of dataset) + a handful of corrupt rows where item text
# leaked into the Category column.
KEEP = {'Services', 'Info', 'Data', 'Information', 'Counterfeits', 'Forgeries',
        'Weapons', 'Electronics', 'Tobacco', 'Jewelry', 'Chemicals', 'Other',
        'Drug paraphernalia'}
df['top'] = df['Category'].str.split('/').str[0]
df = df[df['top'].isin(KEEP)].copy()

# 'Info' and 'Information' are duplicates from different scrape batches
df['top'] = df['top'].replace({'Information': 'Info'})

# --- Build the text field ----------------------------------------------
df['text'] = (df['Item'].fillna('') + ' . ' + df['Item Description'].fillna('')).str.strip()
df = df[df['text'].str.len() > 10]

# Deduplicate — the same listing often appears on multiple dates
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'After filter + dedup: {len(df)} rows')

# --- Label encoding ----------------------------------------------------
label_names = sorted(df['top'].unique())
label2id = {n: i for i, n in enumerate(label_names)}
df['label'] = df['top'].map(label2id)
print(f'\n{len(label_names)} classes:')
print(df['top'].value_counts().to_string())

lens = df['text'].str.split().str.len()
print(f'\nText length (words): median={int(lens.median())}, '
      f'p95={int(lens.quantile(0.95))}, max={lens.max()}')

# --- 80/10/10 split, stratified by label -------------------------------
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

features = Features({
    'text': Value('string'),
    'label': ClassLabel(names=label_names),
})
ds = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']], features=features, preserve_index=False),
    'validation': Dataset.from_pandas(val_df[['text', 'label']], features=features, preserve_index=False),
    'test': Dataset.from_pandas(test_df[['text', 'label']], features=features, preserve_index=False),
})
ds.save_to_disk(str(PROCESSED / 'agora_listings'))
with open(PROCESSED / 'agora_listings' / 'label_names.json', 'w') as f:
    json.dump({'names': label_names, 'label2id': label2id}, f, indent=2)

print(f'\nSaved: {ds}')

Raw rows: 109689
After filter + dedup: 16219 rows

12 classes:
top
Info                  3853
Services              2578
Counterfeits          2331
Data                  2021
Other                 1413
Forgeries             1040
Drug paraphernalia     839
Weapons                656
Electronics            597
Jewelry                417
Tobacco                384
Chemicals               90

Text length (words): median=38, p95=50, max=4367


Saving the dataset (0/1 shards):   0%|          | 0/12975 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1622 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1622 [00:00<?, ? examples/s]


Saved: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 12975
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1622
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1622
    })
})


**What to notice:** after filtering, the dataset is 10× smaller than the raw 110k, but it's also 10× more useful — these categories are what dark web CTI is actually about. Compare the class balance here to the raw dump (where Drugs was 85% of everything) — this is a much cleaner learning signal.

## 2 — Tokenize with ModernBERT

Listings are short (p95 ≈ 50 words), so `MAX_LEN=256` covers essentially everything and lets us use big batches on an 8 GB GPU.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_from_disk

assert torch.cuda.is_available(), 'This notebook expects a GPU. See nb 01 for a CPU variant.'
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

MODEL_CKPT = 'answerdotai/ModernBERT-base'
MAX_LEN = 256

ds = load_from_disk(str(PROCESSED / 'agora_listings'))
with open(PROCESSED / 'agora_listings' / 'label_names.json') as f:
    vocab = json.load(f)
label_names = vocab['names']
num_labels = len(label_names)

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

tok = ds.map(tokenize, batched=True, remove_columns=['text'])
print(tok)

GPU: NVIDIA GeForce RTX 3070 Laptop GPU (8.2 GB)


Map:   0%|          | 0/12975 [00:00<?, ? examples/s]

Map:   0%|          | 0/1622 [00:00<?, ? examples/s]

Map:   0%|          | 0/1622 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 12975
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1622
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1622
    })
})


## 3 — Fine-tune on GPU

Multi-class (single label per listing) — so `problem_type` is the default single-label softmax head, not multi-label like nb 01. With fp16, batch 16, and short sequences, each epoch on ~13k training examples finishes in ~1–2 minutes on an RTX 3070.

In [3]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=num_labels,
    id2label={i: n for i, n in enumerate(label_names)},
    label2id={n: i for i, n in enumerate(label_names)},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0),
    }

args = TrainingArguments(
    output_dir='models/modernbert_agora',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,                  # halves memory, ~1.5x speed on RTX 3070
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok['train'],
    eval_dataset=tok['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.576108,0.595943,0.815043,0.820020,0.815308
2,0.324105,0.519887,0.841554,0.860232,0.841041
3,0.153292,0.607055,0.836621,0.852976,0.835370


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2433, training_loss=0.5282090518683306, metrics={'train_runtime': 381.6461, 'train_samples_per_second': 101.992, 'train_steps_per_second': 6.375, 'total_flos': 2934935813306664.0, 'train_loss': 0.5282090518683306, 'epoch': 3.0})

## 4 — Evaluate on test + per-class breakdown

No threshold tuning here (single-label classification — argmax is final). Instead, we look at the **per-class confusion** — which categories does the model confuse for which? That's the real diagnostic for a marketplace classifier.

In [4]:
test_out = trainer.predict(tok['test'])
y_true = test_out.label_ids
y_pred = np.argmax(test_out.predictions, axis=-1)

print(f'Test accuracy:    {accuracy_score(y_true, y_pred):.4f}')
print(f'Test macro F1:    {f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
print(f'Test weighted F1: {f1_score(y_true, y_pred, average="weighted", zero_division=0):.4f}')

print('\nPer-class breakdown:')
per = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0,
                                       labels=list(range(num_labels)))
for i, name in enumerate(label_names):
    print(f'  {name:22s}  P={per[0][i]:.3f}  R={per[1][i]:.3f}  F1={per[2][i]:.3f}  n={per[3][i]}')

# Confusion matrix — which classes does the model mix up?
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)))
print(f'\nConfusion matrix (rows=true, cols=predicted):')
print(' ' * 22 + ' '.join(f'{n[:6]:>7s}' for n in label_names))
for i, name in enumerate(label_names):
    print(f'  {name:20s}  ' + ' '.join(f'{cm[i,j]:7d}' for j in range(num_labels)))

Test accuracy:    0.8354
Test macro F1:    0.8364
Test weighted F1: 0.8360

Per-class breakdown:
  Chemicals               P=0.857  R=0.667  F1=0.750  n=9
  Counterfeits            P=0.936  R=0.944  F1=0.940  n=233
  Data                    P=0.876  R=0.837  F1=0.856  n=202
  Drug paraphernalia      P=0.888  R=0.940  F1=0.913  n=84
  Electronics             P=0.759  R=0.746  F1=0.752  n=59
  Forgeries               P=0.978  R=0.837  F1=0.902  n=104
  Info                    P=0.826  R=0.875  F1=0.850  n=385
  Jewelry                 P=0.868  R=0.786  F1=0.825  n=42
  Other                   P=0.704  R=0.669  F1=0.686  n=142
  Services                P=0.703  R=0.760  F1=0.730  n=258
  Tobacco                 P=1.000  R=0.872  F1=0.932  n=39
  Weapons                 P=0.965  R=0.846  F1=0.902  n=65

Confusion matrix (rows=true, cols=predicted):
                       Chemic  Counte    Data  Drug p  Electr  Forger    Info  Jewelr   Other  Servic  Tobacc  Weapon
  Chemicals              

## 5 — Save + inference on live examples

Save the model, then sanity-check on a few handwritten listings that look like something you might actually see scraped off a marketplace.

In [5]:
FINAL = Path('models/modernbert_agora_final')
FINAL.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL))
tokenizer.save_pretrained(str(FINAL))
with open(FINAL / 'label_names.json', 'w') as f:
    json.dump({'names': label_names, 'label2id': vocab['label2id']}, f, indent=2)
print(f'Saved to {FINAL}')

# --- Inference on handwritten examples --------------------------------
examples = [
    'US CC Fullz with CVV and DOB - 100 pack bulk discount, fresh from 2025 breach',
    'Netflix premium accounts 1 year warranty, working worldwide',
    'Glock 19 9mm with 2 magazines, serial filed, ships EU-only via dead drop',
    'Fake UK drivers license holographic - photo + signature, 48hr turnaround',
    'How to cash out bitcoin anonymously - 40 page PDF guide with step by step',
    'MacBook Pro 16 M1 - sealed box, cashapp only, no returns',
]

model.eval()
device = next(model.parameters()).device
print('\nLive inference:')
for ex in examples:
    enc = tokenizer(ex, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits[0], dim=-1).cpu().numpy()
    top3 = probs.argsort()[-3:][::-1]
    print(f'\n  {ex[:70]}...')
    for i in top3:
        print(f'    {label_names[i]:22s}  {probs[i]:.3f}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to models/modernbert_agora_final

Live inference:

  US CC Fullz with CVV and DOB - 100 pack bulk discount, fresh from 2025...
    Services                0.480
    Other                   0.391
    Info                    0.029

  Netflix premium accounts 1 year warranty, working worldwide...
    Services                0.638
    Data                    0.332
    Other                   0.017

  Glock 19 9mm with 2 magazines, serial filed, ships EU-only via dead dr...
    Weapons                 0.999
    Other                   0.001
    Services                0.000

  Fake UK drivers license holographic - photo + signature, 48hr turnarou...
    Forgeries               0.999
    Counterfeits            0.001
    Services                0.000

  How to cash out bitcoin anonymously - 40 page PDF guide with step by s...
    Info                    0.995
    Services                0.004
    Other                   0.000

  MacBook Pro 16 M1 - sealed box, cashapp only, no returns.

## What you've done

- Built a **CTI-relevant slice** of the Agora marketplace dump (16k listings across 12 non-drug categories) — the part that matters for threat intel, not the drug-heavy tail.
- Fine-tuned ModernBERT on GPU with fp16 — orders of magnitude faster than nb 01's CPU run, and the short-text regime means we don't need long context here.
- Evaluated per-class and inspected the confusion matrix to see **which categories the model genuinely confuses** (Info ↔ Data is the interesting one — both are digital fraud content).

## What's next in CTI 301

- **Notebook 03: domain-pretrained comparison.** Same data, swap in **CySecBERT** / **SecureBERT** / **DarkBERT** (if access granted). The question: does security-domain or dark-web-domain pretraining beat generic ModernBERT on this task, or is the fine-tuning signal strong enough that pretraining doesn't matter?
- **Notebook 04: IOC & TTP extraction.** Move from whole-listing classification to span-level extraction of CVEs, wallet addresses, email dumps, etc. Using GLiNER for zero-shot, then fine-tuning on the auto-labeled data.